# Integration Layer Data Quality Validation

## Purpose

This notebook validates the relationships between the Silver datasets before creating the Integration layer.

The main goal is to identify unmatched records before joining the datasets so that no records are silently dropped.

### Relationships being validated

1. Green Taxi pickup location -> Taxi Zone
2. Green Taxi dropoff location -> Taxi Zone
3. Green Taxi pickup hour -> Weather hour

The results determine the appropriate join strategy for the Integration layer.

### How results are recorded

This notebook is **evidence**, not a gate. Each check returns its row and
persists nothing.

`docs/validation.md` points every check at
`` `ftw-week-08`.`01-control`.`data_quality_results` ``, and
`etl/README.md` states that stage 04 creates no tables of its own. Persisting
these rows is the job of `etl/04_integration/90_validate_integration.sql`,
which already references that table. Writing them from here as well would
create a second source of truth.

### Status and severity

Both follow the shared result contract in `docs/validation.md`:

- `severity` is the policy for the check: `INFO`, `WARN` or `FAIL`.
- `status` is the outcome of this run: `INFO`, `PASS`, `WARN` or `FAIL`.
- A `WARN`-severity check whose `fail_pct` exceeds `threshold_pct` becomes `FAIL`.
- Thresholds are documented tolerances, not values tuned to today's data.
- Every check defines what happens when its input is empty. An empty dataset
  never passes silently.

In [0]:
%sql
-- One run id AND one run timestamp, shared by every check in this run, so the
-- three rows below can be read as a single run. current_timestamp() is
-- evaluated once per cell, so a bare call in the summary cell would stamp the
-- time that cell ran rather than the time the run started. Both are variables
-- for that reason.
DECLARE OR REPLACE VARIABLE dq_run_id STRING;
DECLARE OR REPLACE VARIABLE dq_run_at TIMESTAMP;

SET VARIABLE dq_run_id = uuid();
SET VARIABLE dq_run_at = current_timestamp();

SELECT
    dq_run_id AS run_id,
    dq_run_at AS executed_at;

## 1. Pickup Location -> Taxi Zone

### What are we checking?

Each Green Taxi trip contains a `pickup_location_id`.

We check whether this ID exists in the Taxi Zone Silver table.

### Expected result

We expect every pickup location ID to have a matching Taxi Zone record.

An unmatched record would mean that the taxi trip's pickup location cannot be
linked to the zone dimension.

### How the count is taken

`total_count` is the **trip** count, taken before the join. If
`taxi_zones_clean` ever holds two rows for one `location_id`, each affected
trip duplicates in the join output; counting after the join would inflate the
denominator and understate `fail_pct` while still reporting zero unmatched.
`fan_out_rows` is the difference, and any non-zero value fails the check.

Two kinds of failure are separated, because they have different causes:

- `missing_source_id` -- the trip has no `pickup_location_id` at all
- `unmatched_location_id` -- the ID is present but no zone carries it

IDs 264 (unknown) and 265 (outside_nyc) are valid Silver members per D16, so
they match and are not failures.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW chk_pickup_zone AS
WITH trips AS (
    SELECT pickup_location_id AS location_id
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean
),

resolved AS (
    SELECT
        t.location_id,
        z.location_id AS zone_location_id
    FROM trips t
    LEFT JOIN `ftw-week-08`.`03-silver`.taxi_zones_clean z
        ON t.location_id = z.location_id
),

-- Both CTEs are aggregates with no GROUP BY, so each returns exactly one row
-- even when its input is empty. That is what makes the empty-input branch
-- below reachable: under a GROUP BY, an empty input yields no rows at all and
-- the check would report nothing instead of failing.
totals AS (
    SELECT COUNT(*) AS total_count
    FROM trips
),

counted AS (
    SELECT
        COUNT(*)                                                            AS join_rows,
        COUNT_IF(r.zone_location_id IS NOT NULL)                            AS matched,
        COUNT_IF(r.location_id IS NULL)                                     AS missing_source_id,
        COUNT_IF(r.location_id IS NOT NULL AND r.zone_location_id IS NULL)  AS unmatched_location_id
    FROM resolved r
)

SELECT
    'integration'                                    AS layer,
    'green_taxi_to_taxi_zone'                        AS dataset,
    'pickup_location_zone_match'                     AS check_name,
    'referential_integrity'                          AS check_type,
    'FAIL'                                           AS severity,
    CAST(0.0 AS DOUBLE)                              AS threshold_pct,
    total_count,
    matched,
    missing_source_id,
    unmatched_location_id,
    join_rows - total_count                          AS fan_out_rows,
    missing_source_id + unmatched_location_id        AS fail_count,
    -- DECIMAL division widens the scale, which prints zero as 0E-14 and does
    -- not match the DOUBLE that data_quality_results declares. Round for the
    -- recorded value; the threshold comparison below stays on full precision.
    CAST(ROUND(COALESCE(
        try_divide((missing_source_id + unmatched_location_id) * 100.0, total_count),
        0.0
    ), 6) AS DOUBLE)                                 AS fail_pct,
    CASE
        WHEN total_count = 0                              THEN 'FAIL'
        WHEN join_rows <> total_count                     THEN 'FAIL'
        WHEN missing_source_id + unmatched_location_id = 0 THEN 'PASS'
        ELSE 'FAIL'
    END                                              AS status,
    CONCAT(
        'trips=',                  CAST(total_count AS STRING),
        '; matched=',              CAST(matched AS STRING),
        '; missing_source_id=',    CAST(missing_source_id AS STRING),
        '; unmatched_location_id=', CAST(unmatched_location_id AS STRING),
        '; fan_out_rows=',         CAST(join_rows - total_count AS STRING)
    )                                                AS details
FROM counted CROSS JOIN totals;

SELECT * FROM chk_pickup_zone;

## 2. Dropoff Location -> Taxi Zone

### What are we checking?

Each Green Taxi trip contains a `dropoff_location_id`.

We check whether this ID exists in the Taxi Zone Silver table.

### Expected result

We expect every dropoff location ID to have a matching Taxi Zone record.

The method is identical to check 1. Pickup and dropoff are kept as separate
checks because they are separate roles against the same dimension (D12), and
one can break while the other passes.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW chk_dropoff_zone AS
WITH trips AS (
    SELECT dropoff_location_id AS location_id
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean
),

resolved AS (
    SELECT
        t.location_id,
        z.location_id AS zone_location_id
    FROM trips t
    LEFT JOIN `ftw-week-08`.`03-silver`.taxi_zones_clean z
        ON t.location_id = z.location_id
),

-- Both CTEs are aggregates with no GROUP BY, so each returns exactly one row
-- even when its input is empty. That is what makes the empty-input branch
-- below reachable: under a GROUP BY, an empty input yields no rows at all and
-- the check would report nothing instead of failing.
totals AS (
    SELECT COUNT(*) AS total_count
    FROM trips
),

counted AS (
    SELECT
        COUNT(*)                                                            AS join_rows,
        COUNT_IF(r.zone_location_id IS NOT NULL)                            AS matched,
        COUNT_IF(r.location_id IS NULL)                                     AS missing_source_id,
        COUNT_IF(r.location_id IS NOT NULL AND r.zone_location_id IS NULL)  AS unmatched_location_id
    FROM resolved r
)

SELECT
    'integration'                                    AS layer,
    'green_taxi_to_taxi_zone'                        AS dataset,
    'dropoff_location_zone_match'                    AS check_name,
    'referential_integrity'                          AS check_type,
    'FAIL'                                           AS severity,
    CAST(0.0 AS DOUBLE)                              AS threshold_pct,
    total_count,
    matched,
    missing_source_id,
    unmatched_location_id,
    join_rows - total_count                          AS fan_out_rows,
    missing_source_id + unmatched_location_id        AS fail_count,
    -- DECIMAL division widens the scale, which prints zero as 0E-14 and does
    -- not match the DOUBLE that data_quality_results declares. Round for the
    -- recorded value; the threshold comparison below stays on full precision.
    CAST(ROUND(COALESCE(
        try_divide((missing_source_id + unmatched_location_id) * 100.0, total_count),
        0.0
    ), 6) AS DOUBLE)                                 AS fail_pct,
    CASE
        WHEN total_count = 0                              THEN 'FAIL'
        WHEN join_rows <> total_count                     THEN 'FAIL'
        WHEN missing_source_id + unmatched_location_id = 0 THEN 'PASS'
        ELSE 'FAIL'
    END                                              AS status,
    CONCAT(
        'trips=',                  CAST(total_count AS STRING),
        '; matched=',              CAST(matched AS STRING),
        '; missing_source_id=',    CAST(missing_source_id AS STRING),
        '; unmatched_location_id=', CAST(unmatched_location_id AS STRING),
        '; fan_out_rows=',         CAST(join_rows - total_count AS STRING)
    )                                                AS details
FROM counted CROSS JOIN totals;

SELECT * FROM chk_dropoff_zone;

## 3. Trip -> Weather Hour

### What are we checking?

Each Green Taxi trip is matched to the weather observation for its pickup hour.

### Matched in UTC, on the pinned coordinate and model

The match key is the one `etl/05_gold/30_fact_taxi_trip.sql` uses:

```
w.coordinate_id            = 'nyc_approved_point'
w.weather_model            = 'era5'
w.observation_timestamp_utc = date_trunc('hour', to_utc_timestamp(pickup_datetime_local, 'America/New_York'))
```

Three reasons, all of which change the number this check reports:

1. **Gold matches in UTC.** A check that matches local-to-local measures a
   different join from the one the fact table will perform, so its coverage
   figure would not be the coverage Gold gets.
2. **`weather_hourly`'s business key is (coordinate_id, observation_timestamp_utc,
   weather_model).** Matching on the hour alone would silently collapse a second
   coordinate or model into the same hour the moment either is ingested.
3. **Local hours are not unique across a DST fall-back.** One local hour then
   carries two observations. Matching in UTC avoids the problem instead of
   hiding it behind a `DISTINCT`.

Weather uniqueness is now checked rather than assumed: `weather_rows` and
`weather_hours` must be equal, and `fan_out_rows` must be zero.

### Expected result

The two datasets have different coverage windows, so some unmatched records are
expected. Unmatched trips are classified against the weather table's **actual**
minimum and maximum observation, read at run time:

- `before_coverage` -- pickup earlier than the first weather observation
- `after_coverage` -- pickup later than the last weather observation
- `hole_inside_coverage` -- pickup inside the window but no observation for that
  hour, which would be a genuine gap in the weather data
- `invalid_pickup_timestamp` -- no usable pickup timestamp
- `unclassified_unmatched` -- unmatched with no coverage window to compare
  against, i.e. the weather table is empty

`unaccounted_rows` must be zero: the buckets and `matched` have to add up to the
join output, so no row escapes classification.

### Threshold

`0.05%` is a tolerance for trips that genuinely fall outside the weather window
-- stray old pickup timestamps arriving inside a later month's file. It is
deliberately **not** set to the currently observed rate. If `fail_pct` exceeds
it, the check fails and the Integration gate stays shut until the cause is
fixed. See the closing analysis cell.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW chk_trip_weather AS
WITH weather AS (
    SELECT observation_timestamp_utc
    FROM `ftw-week-08`.`03-silver`.weather_hourly
    WHERE coordinate_id = 'nyc_approved_point'
      AND weather_model = 'era5'
),

-- Aggregate with no GROUP BY: one row even when weather is empty, in which
-- case the bounds are NULL and the guards below catch it.
coverage AS (
    SELECT
        MIN(observation_timestamp_utc)            AS cov_start_utc,
        MAX(observation_timestamp_utc)            AS cov_end_utc,
        COUNT(*)                                  AS weather_rows,
        COUNT(DISTINCT observation_timestamp_utc) AS weather_hours
    FROM weather
),

trips AS (
    SELECT to_utc_timestamp(pickup_datetime_local, 'America/New_York') AS pickup_ts_utc
    FROM `ftw-week-08`.`03-silver`.green_taxi_clean
),

resolved AS (
    SELECT
        t.pickup_ts_utc,
        c.cov_start_utc,
        c.cov_end_utc,
        w.observation_timestamp_utc
    FROM trips t
    CROSS JOIN coverage c
    LEFT JOIN weather w
        ON w.observation_timestamp_utc = date_trunc('hour', t.pickup_ts_utc)
),

totals AS (
    SELECT COUNT(*) AS total_count
    FROM trips
),

-- Aggregate with no GROUP BY, so an empty trips table still produces one row
-- here and the guards below can act on it.
counted AS (
    SELECT
        COUNT(*)                              AS join_rows,
        COUNT_IF(r.observation_timestamp_utc IS NOT NULL) AS matched,
        COUNT_IF(r.pickup_ts_utc IS NULL)                 AS invalid_pickup_timestamp,
        COUNT_IF(r.observation_timestamp_utc IS NULL
                 AND r.pickup_ts_utc IS NOT NULL
                 AND r.pickup_ts_utc < r.cov_start_utc)   AS before_coverage,
        COUNT_IF(r.observation_timestamp_utc IS NULL
                 AND r.pickup_ts_utc IS NOT NULL
                 AND r.pickup_ts_utc > r.cov_end_utc)     AS after_coverage,
        COUNT_IF(r.observation_timestamp_utc IS NULL
                 AND r.pickup_ts_utc IS NOT NULL
                 AND r.pickup_ts_utc >= r.cov_start_utc
                 AND r.pickup_ts_utc <= r.cov_end_utc)    AS hole_inside_coverage,
        COUNT_IF(r.observation_timestamp_utc IS NULL
                 AND r.pickup_ts_utc IS NOT NULL
                 AND (r.cov_start_utc IS NULL OR r.cov_end_utc IS NULL)) AS unclassified_unmatched
    FROM resolved r
),

scored AS (
    SELECT
        c.*,
        t.total_count,
        v.weather_rows,
        v.weather_hours,
        v.cov_start_utc,
        v.cov_end_utc,
        c.invalid_pickup_timestamp + c.before_coverage + c.after_coverage
            + c.hole_inside_coverage + c.unclassified_unmatched AS fail_count,
        c.join_rows - (c.matched + c.invalid_pickup_timestamp + c.before_coverage
            + c.after_coverage + c.hole_inside_coverage + c.unclassified_unmatched) AS unaccounted_rows
    FROM counted c
    CROSS JOIN totals t
    CROSS JOIN coverage v
)

SELECT
    'integration'                            AS layer,
    'green_taxi_to_weather'                  AS dataset,
    'trip_weather_hour_match'                AS check_name,
    'referential_integrity'                  AS check_type,
    'WARN'                                   AS severity,
    CAST(0.05 AS DOUBLE)                     AS threshold_pct,
    total_count,
    matched,
    invalid_pickup_timestamp,
    before_coverage,
    after_coverage,
    hole_inside_coverage,
    unclassified_unmatched,
    join_rows - total_count                  AS fan_out_rows,
    unaccounted_rows,
    weather_rows,
    weather_hours,
    from_utc_timestamp(cov_start_utc, 'America/New_York') AS coverage_start_local,
    from_utc_timestamp(cov_end_utc,   'America/New_York') AS coverage_end_local,
    fail_count,
    -- Rounded and cast for the same reason as the zone checks; the CASE
    -- below still compares the unrounded value.
    CAST(ROUND(COALESCE(try_divide(fail_count * 100.0, total_count), 0.0), 6) AS DOUBLE) AS fail_pct,
    CASE
        WHEN total_count = 0                 THEN 'FAIL'  -- no trips to check
        WHEN weather_rows = 0                THEN 'FAIL'  -- no weather to match against
        WHEN weather_rows <> weather_hours   THEN 'FAIL'  -- weather not unique per hour
        WHEN join_rows <> total_count        THEN 'FAIL'  -- the join fanned out
        WHEN unaccounted_rows <> 0           THEN 'FAIL'  -- buckets do not add up
        WHEN fail_count = 0                  THEN 'PASS'
        WHEN COALESCE(try_divide(fail_count * 100.0, total_count), 0.0) > 0.05 THEN 'FAIL'
        ELSE 'WARN'
    END                                      AS status,
    CONCAT(
        'trips=',                    CAST(total_count AS STRING),
        '; matched=',                CAST(matched AS STRING),
        '; before_coverage=',        CAST(before_coverage AS STRING),
        '; after_coverage=',         CAST(after_coverage AS STRING),
        '; hole_inside_coverage=',   CAST(hole_inside_coverage AS STRING),
        '; invalid_pickup_timestamp=', CAST(invalid_pickup_timestamp AS STRING),
        '; weather_coverage_local=[',
            CAST(from_utc_timestamp(cov_start_utc, 'America/New_York') AS STRING), ' .. ',
            CAST(from_utc_timestamp(cov_end_utc,   'America/New_York') AS STRING), ']',
        '; fan_out_rows=',           CAST(join_rows - total_count AS STRING)
    )                                        AS details
FROM scored;

SELECT * FROM chk_trip_weather;

## Run summary

The three checks in the shared result contract's columns. This is the shape
`etl/04_integration/90_validate_integration.sql` will persist to
`` `ftw-week-08`.`01-control`.`data_quality_results` `` once that table exists.

`batch_id`, `source_version_id`, `code_revision` and `evidence_location` are
part of that contract and are **not** produced here -- this notebook does not
persist rows, and inventing the values would make them untraceable. The gate
file supplies them.

In [0]:
%sql
SELECT dq_run_id AS run_id, dq_run_at AS executed_at,
       layer, dataset, check_name, check_type,
       severity, status, fail_count, total_count, fail_pct, threshold_pct, details
FROM chk_pickup_zone

UNION ALL
SELECT dq_run_id AS run_id, dq_run_at AS executed_at,
       layer, dataset, check_name, check_type,
       severity, status, fail_count, total_count, fail_pct, threshold_pct, details
FROM chk_dropoff_zone

UNION ALL
SELECT dq_run_id AS run_id, dq_run_at AS executed_at,
       layer, dataset, check_name, check_type,
       severity, status, fail_count, total_count, fail_pct, threshold_pct, details
FROM chk_trip_weather

ORDER BY check_name;

## Analysis of Integration DQ Results

Numbers below are from the run recorded in this notebook's cell outputs.

### Summary

| Relationship | Trips | Matched | Unmatched | Unmatched % | Status |
|---|---:|---:|---:|---:|---|
| Pickup -> Taxi Zone | 133,353 | 133,353 | 0 | 0.00% | PASS |
| Dropoff -> Taxi Zone | 133,353 | 133,353 | 0 | 0.00% | PASS |
| Trip -> Weather Hour | 133,353 | 133,173 | 180 | 0.13% | FAIL |

### Taxi Zone relationships

Every pickup and dropoff location ID matched the Taxi Zone dimension, with no
missing source IDs and no fan-out. These relationships create no record loss.

### Weather relationship: the 175 are an ingestion defect, not a coverage limit

180 trips had no weather observation, split by the check into **5
`before_coverage`** and **175 `after_coverage`**. The 175 have a root cause that
is fixable upstream:

`etl/02_bronze/20_load_open_meteo.sql` requests `2026-03-01` to `2026-05-31`
with **no timezone parameter**, so Open-Meteo returns GMT. D09 records exactly
that from the profiled response: `utc_offset_seconds = 0`, `timezone = GMT`. The
response therefore covers `2026-03-01 00:00` to `2026-05-31 23:00` **UTC**.

Silver converts with `from_utc_timestamp(..., 'America/New_York')`. May 31 2026
is EDT (UTC-4), so:

```
2026-05-31 23:00 UTC  ->  2026-05-31 19:00 local
```

Local weather coverage ends at 19:00 on May 31, leaving local hours 20:00, 21:00,
22:00 and 23:00 with no observation. Four hours -- the 175 trips.

### The weather series itself is complete

`weather_rows` and `weather_hours` are both **2208**, which is exactly 92 days
x 24 hours (744 + 720 + 744 for March, April and May). No hour is missing, none
is duplicated, and `hole_inside_coverage` is 0, so there are no gaps inside the
window either.

The window is therefore the right **length** and the wrong **offset**: 2208
hours spanning `2026-02-28 19:00` to `2026-05-31 19:00` local, where the
intended window was `2026-03-01 00:00` to `2026-05-31 23:00` local. Re-requesting
with `timezone=America/New_York` moves the same 2208-hour series onto the
intended local window. Nothing else needs repairing, which is what makes this a
one-line Bronze fix rather than a data recovery job.

The request window is short by the UTC offset at its tail. Re-fetching with
`timezone=America/New_York`, or with `end_date=2026-06-01` and clipping, removes
175 of the 180. Tracked as a separate issue: it is a Bronze change and requires
re-landing the source JSON.

The same missing parameter is the `requested_timezone` gap already flagged in
`etl/03_silver/20_clean_weather_hourly.sql` and in D18's contract gaps. All
three are one fix.

### The remaining 5 are genuine

Coverage **starts** at `2026-02-28 19:00` local -- the same offset, working in
our favour at the front of the window. The 5 `before_coverage` trips are earlier
than that: stray old pickup timestamps inside the March file, i.e. real
late-arriving data. They are a legitimate source trait and sit under the 0.05%
tolerance once the 175 are fixed.

### Why this check reports FAIL

`fail_pct` of 0.13% is above the 0.05% tolerance, so the Integration gate is
shut. That is intended. Setting the threshold to 0.13% would make a known,
fixable ingestion defect look green, and `docs/validation.md` requires
thresholds to be documented tolerances rather than values fitted to today's
data.

Expected once the weather re-fetch lands: `after_coverage` drops to 0,
`fail_count` to 5, `fail_pct` to roughly 0.004%, and status to `WARN` with the
5 records explained.

### Known limitation: DST ambiguity from November onward

Silver stores taxi times as local wall-clock with no UTC offset, so
`to_utc_timestamp` has to resolve local labels that are not unique. March to May
contains only a spring-forward, where the affected local hour does not exist and
nothing is ambiguous -- `invalid_pickup_timestamp` is 0 for this run.

At the November fall-back, one local hour label maps to two real instants and
the conversion has to pick one, so some trips could match the neighbouring
weather hour. This is a limitation of the source, which does not record an
offset, and cannot be fixed in this check. Recording it here so it is not a
surprise when the reporting window extends past October.

### Integration Decision

`LEFT JOIN` from the Green Taxi dataset, for both relationships.

Trips are retained even when the related record is unavailable:

- the trip stays in the Integration dataset
- related fields are `NULL`
- an explicit match-status field records why

Match-status vocabularies, matching stage 04's contract and the Gold fact:

- zones: `matched_regular`, `matched_special`, `missing_source_id`, `unmatched_location_id`
- weather: `matched_unique`, `no_match`, `invalid_pickup_timestamp`

To be recorded as a numbered decision in `docs/decisions.md` before the
Integration SQL is written.